#### importing libraries


In [3]:

import numpy as np
import pandas as pd


#### Extraction Layer

In [4]:
healthcare_df = pd.read_csv(r'dataset\raw_dataset\health_dataset.csv')
healthcare_df.head()

,patient_id,age,gender,bmi,systolic_bp,diastolic_bp,heart_rate,cholesterol_mg_dl,glucose_mg_dl,smoker,diabetes,hypertension,admission_type,primary_diagnosis,admission_date
0,1,69,Other,22.0,174,79,66,142,93,No,Yes,No,Outpatient,Stroke,09:36.9
1,2,32,Other,27.3,125,108,76,220,100,Yes,Yes,No,Emergency,Heart Disease,09:36.9
2,3,89,Other,23.5,169,86,99,122,168,No,No,No,Emergency,Anxiety,09:36.9
3,4,78,Female,26.0,122,60,61,257,199,No,No,No,Elective,Diabetes,09:36.9
4,5,38,Male,31.7,148,96,117,176,154,Yes,No,No,Emergency,Hypertension,09:36.9


In [5]:
healthcare_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   patient_id         20000 non-null  int64  
 1   age                20000 non-null  int64  
 2   gender             20000 non-null  object 
 3   bmi                20000 non-null  float64
 4   systolic_bp        20000 non-null  int64  
 5   diastolic_bp       20000 non-null  int64  
 6   heart_rate         20000 non-null  int64  
 7   cholesterol_mg_dl  20000 non-null  int64  
 8   glucose_mg_dl      20000 non-null  int64  
 9   smoker             20000 non-null  object 
 10  diabetes           20000 non-null  object 
 11  hypertension       20000 non-null  object 
 12  admission_type     20000 non-null  object 
 13  primary_diagnosis  20000 non-null  object 
 14  admission_date     20000 non-null  object 
dtypes: float64(1), int64(7), object(7)
memory usage: 2.3+ MB


In [6]:
healthcare_df.columns

Index(['patient_id', 'age', 'gender', 'bmi', 'systolic_bp', 'diastolic_bp',
       'heart_rate', 'cholesterol_mg_dl', 'glucose_mg_dl', 'smoker',
       'diabetes', 'hypertension', 'admission_type', 'primary_diagnosis',
       'admission_date'],
      dtype='object')

#### Data Cleaning and Transformation

# convert admission_date from string to datetime object

In [7]:
healthcare_df["admission_date"] = pd.to_datetime(healthcare_df["admission_date"])

In [8]:
healthcare_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   patient_id         20000 non-null  int64         
 1   age                20000 non-null  int64         
 2   gender             20000 non-null  object        
 3   bmi                20000 non-null  float64       
 4   systolic_bp        20000 non-null  int64         
 5   diastolic_bp       20000 non-null  int64         
 6   heart_rate         20000 non-null  int64         
 7   cholesterol_mg_dl  20000 non-null  int64         
 8   glucose_mg_dl      20000 non-null  int64         
 9   smoker             20000 non-null  object        
 10  diabetes           20000 non-null  object        
 11  hypertension       20000 non-null  object        
 12  admission_type     20000 non-null  object        
 13  primary_diagnosis  20000 non-null  object        
 14  admiss

In [9]:
#patient table
patient_df = healthcare_df[['patient_id', 'age', 'gender', 'bmi']].copy().drop_duplicates().reset_index(drop=True)
patient_df.head()

,patient_id,age,gender,bmi
0,1,69,Other,22.0
1,2,32,Other,27.3
2,3,89,Other,23.5
3,4,78,Female,26.0
4,5,38,Male,31.7


In [10]:
#Admission table
admission_df = healthcare_df[['patient_id', 'admission_type', 'admission_date', 'primary_diagnosis']].copy().drop_duplicates().reset_index(drop=True)
admission_df.index.name = 'admission_id'
admission_df = admission_df.reset_index()

admission_df.head()

,admission_id,patient_id,admission_type,admission_date,primary_diagnosis
0,0,1,Outpatient,2026-03-05 09:36:54,Stroke
1,1,2,Emergency,2026-03-05 09:36:54,Heart Disease
2,2,3,Emergency,2026-03-05 09:36:54,Anxiety
3,3,4,Elective,2026-03-05 09:36:54,Diabetes
4,4,5,Emergency,2026-03-05 09:36:54,Hypertension


In [11]:
# add `admission_id` back to the main dataframe for downstream tables
healthcare_df = healthcare_df.merge(admission_df[['admission_id', 'patient_id', 'admission_date', 'primary_diagnosis', 'admission_type']], on=['patient_id', 'admission_date', 'primary_diagnosis', 'admission_type'], how='left')
healthcare_df.head()

,patient_id,age,gender,bmi,systolic_bp,diastolic_bp,heart_rate,cholesterol_mg_dl,glucose_mg_dl,smoker,diabetes,hypertension,admission_type,primary_diagnosis,admission_date,admission_id
0,1,69,Other,22.0,174,79,66,142,93,No,Yes,No,Outpatient,Stroke,2026-03-05 09:36:54,0
1,2,32,Other,27.3,125,108,76,220,100,Yes,Yes,No,Emergency,Heart Disease,2026-03-05 09:36:54,1
2,3,89,Other,23.5,169,86,99,122,168,No,No,No,Emergency,Anxiety,2026-03-05 09:36:54,2
3,4,78,Female,26.0,122,60,61,257,199,No,No,No,Elective,Diabetes,2026-03-05 09:36:54,3
4,5,38,Male,31.7,148,96,117,176,154,Yes,No,No,Emergency,Hypertension,2026-03-05 09:36:54,4


In [12]:
#condition table
condition_df = healthcare_df[['patient_id', 'diabetes', 'hypertension', 'smoker', 'cholesterol_mg_dl', 'glucose_mg_dl' ]].copy().drop_duplicates().reset_index(drop=True)
condition_df.index.name = 'condition_id'
condition_df = condition_df.reset_index()
condition_df.head()

,condition_id,patient_id,diabetes,hypertension,smoker,cholesterol_mg_dl,glucose_mg_dl
0,0,1,Yes,No,No,142,93
1,1,2,Yes,No,Yes,220,100
2,2,3,No,No,No,122,168
3,3,4,No,No,No,257,199
4,4,5,No,No,Yes,176,154


In [14]:
#vital table
vital_df = healthcare_df[['admission_id', 'systolic_bp', 'diastolic_bp', 'heart_rate' ]].copy().drop_duplicates().reset_index(drop=True)
vital_df.index.name = 'vital_id'
vital_df = vital_df.reset_index()
vital_df.head()

,vital_id,admission_id,systolic_bp,diastolic_bp,heart_rate
0,0,0,174,79,66
1,1,1,125,108,76
2,2,2,169,86,99
3,3,3,122,60,61
4,4,4,148,96,117


In [15]:
healthcare_df.columns

Index(['patient_id', 'age', 'gender', 'bmi', 'systolic_bp', 'diastolic_bp',
       'heart_rate', 'cholesterol_mg_dl', 'glucose_mg_dl', 'smoker',
       'diabetes', 'hypertension', 'admission_type', 'primary_diagnosis',
       'admission_date', 'admission_id'],
      dtype='object')